# Itchy — Patch Size Ablation

Patch size is THE key byte-level hyperparameter. Test 1, 2, 3, 4, 6, 8.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**

Expected time: ~25 min

In [ ]:
!pip install -q torch numpy huggingface-hub sentencepiece
import torch, math, time, os, shutil, numpy as np
import torch.nn.functional as F
from torch import nn
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm
print(f'GPU: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda')

In [ ]:
# Data
os.makedirs('data/bytes', exist_ok=True)
def dl(fn, sub, dst):
    if Path(dst).exists(): return
    print(f'  Downloading {fn}...')
    c = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn, subfolder=sub, repo_type='dataset')).resolve())
    try: os.link(c, dst)
    except: shutil.copy2(c, dst)

dl('fineweb_train_000000.bin','datasets/datasets/fineweb10B_sp1024','data/train.bin')
dl('fineweb_val_000000.bin','datasets/datasets/fineweb10B_sp1024','data/val.bin')
dl('fineweb_1024_bpe.model','datasets/tokenizers','data/tok.model')

def load_shard(p):
    h = np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4)
def write_shard(p, d):
    h = np.zeros(256,dtype='<i4'); h[0]=20240520; h[1]=1; h[2]=len(d)
    with open(p,'wb') as f: f.write(h.tobytes()); f.write(d.astype('<u2').tobytes())

if not Path('data/bytes/train.bin').exists():
    sp = spm.SentencePieceProcessor(model_file='data/tok.model')
    for s,d in [('data/train.bin','data/bytes/train.bin'),('data/val.bin','data/bytes/val.bin')]:
        print(f'Converting {s}...')
        tok = load_shard(s)
        bs = [np.frombuffer(sp.decode(tok[i:i+100000].tolist()).encode('utf-8'),dtype=np.uint8)
              for i in range(0,len(tok),100000)]
        write_shard(d, np.concatenate(bs))

def load_bytes(p):
    h = np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4).astype(np.int64)

train_data = load_bytes('data/bytes/train.bin')
val_data = load_bytes('data/bytes/val.bin')
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

In [ ]:
# ============================================================
# MODEL — parameterized by patch_size
# ============================================================

class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None
    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)

def apply_rope(x, cos, sin):
    h = x.size(-1) // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1*cos+x2*sin, x1*(-sin)+x2*cos), dim=-1)

class Attn(nn.Module):
    def __init__(self, dim, nh, nkv, qk_gain):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, dim//nh
        kv = nkv * self.hd
        self.cq = nn.Linear(dim,dim,bias=False)
        self.ck = nn.Linear(dim,kv,bias=False)
        self.cv = nn.Linear(dim,kv,bias=False)
        self.proj = nn.Linear(dim,dim,bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg = nn.Parameter(torch.full((nh,), qk_gain))
        self.rot = Rotary(self.hd)
    def forward(self, x):
        B,S,D = x.shape
        q = self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k = self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v = self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q,k = F.rms_norm(q,(q.size(-1),)), F.rms_norm(k,(k.size(-1),))
        cos,sin = self.rot(S,x.device,q.dtype)
        q,k = apply_rope(q,cos,sin), apply_rope(k,cos,sin)
        q = q * self.qg.to(q.dtype)[None,:,None,None]
        y = F.scaled_dot_product_attention(q,k,v,is_causal=True,enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).contiguous().reshape(B,S,D))

class MLP(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim,dim*mult,bias=False)
        self.proj = nn.Linear(dim*mult,dim,bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        return self.proj(F.leaky_relu(self.fc(x),0.5).square())

class Block(nn.Module):
    def __init__(self, dim, nh, nkv, mlp_mult, qk_gain):
        super().__init__()
        self.attn = Attn(dim,nh,nkv,qk_gain)
        self.mlp = MLP(dim,mlp_mult)
        self.as_ = nn.Parameter(torch.ones(dim))
        self.ms = nn.Parameter(torch.ones(dim))
        self.rm = nn.Parameter(torch.stack((torch.ones(dim),torch.zeros(dim))))
    def forward(self, x, x0):
        m = self.rm.to(x.dtype)
        x = m[0][None,None,:]*x + m[1][None,None,:]*x0
        x = x + self.as_.to(x.dtype)[None,None,:]*self.attn(F.rms_norm(x,(x.size(-1),)))
        x = x + self.ms.to(x.dtype)[None,None,:]*self.mlp(F.rms_norm(x,(x.size(-1),)))
        return x

class Itchy(nn.Module):
    def __init__(self, dim, num_layers, nh, nkv, mlp_mult, patch_size, softcap=30.0):
        super().__init__()
        self.dim, self.ps, self.sc = dim, patch_size, softcap
        self.be = nn.Embedding(260, dim)
        self.pp = nn.Linear(dim*patch_size, dim, bias=False)
        ne = num_layers//2; nd = num_layers-ne
        self.ne, self.nd = ne, nd
        self.sw = nn.Parameter(torch.ones(min(ne,nd), dim))
        self.blocks = nn.ModuleList([Block(dim,nh,nkv,mlp_mult,1.5) for _ in range(num_layers)])
        self.up = nn.Linear(dim, patch_size*260, bias=False)

    def forward(self, ids, tgt):
        B,S = ids.shape; P = self.ps
        x = self.be(ids).reshape(B,S//P,P*self.dim)
        x = F.rms_norm(self.pp(x),(self.dim,))
        x0 = x; skips = []
        for i in range(self.ne): x = self.blocks[i](x,x0); skips.append(x)
        for i in range(self.nd):
            if skips: x = x + self.sw[i].to(x.dtype)[None,None,:]*skips.pop()
            x = self.blocks[self.ne+i](x,x0)
        x = F.rms_norm(x,(x.size(-1),))
        lo = self.up(x).reshape(-1,260)
        lo = self.sc * torch.tanh(lo/self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# TRAINING + EVAL HARNESS
# ============================================================

DIM = 256
LAYERS = 8
NH, NKV = 8, 4
MLP_MULT = 3
LR = 3e-4
STEPS = 3000
# Keep total bytes per batch constant across patch sizes
BATCH_BYTES = 8192

def train_and_val(patch_size):
    # Seq len must be divisible by patch_size. Keep ~512 bytes.
    seq_len = (512 // patch_size) * patch_size
    if seq_len == 0: seq_len = patch_size

    torch.manual_seed(42)
    model = Itchy(DIM, LAYERS, NH, NKV, MLP_MULT, patch_size).to(device).bfloat16()
    p = n_params(model)
    n_patches = seq_len // patch_size

    print(f'\n{"="*60}')
    print(f'Patch size {patch_size}: {p:,} params | seq={seq_len} bytes | {n_patches} patches')
    print(f'{"="*60}')

    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.01)
    pos = 0; t0 = time.time(); losses = []

    for step in range(1, STEPS+1):
        usable = (BATCH_BYTES // seq_len) * seq_len
        if usable == 0: usable = seq_len
        end = pos + usable + 1
        if end > len(train_data): pos = 0; end = usable + 1
        chunk = train_data[pos:end]; pos = end - 1
        x = torch.tensor(chunk[:-1].reshape(-1, seq_len), device=device)
        y = torch.tensor(chunk[1:].reshape(-1, seq_len), device=device)
        opt.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(x, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % 1000 == 0:
            avg = np.mean(losses[-1000:])
            print(f'  step {step} | bpb {avg/math.log(2):.4f} | {time.time()-t0:.0f}s')

    train_time = time.time() - t0

    # Validation
    vl = []
    for i in range(0, min(200*seq_len, len(val_data)-seq_len-1), seq_len):
        chunk = val_data[i:i+seq_len+1]
        x = torch.tensor(chunk[:-1].reshape(1,-1), device=device)
        y = torch.tensor(chunk[1:].reshape(1,-1), device=device)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                vl.append(model(x,y).item())
    val = np.mean(vl)
    bpb = val / math.log(2)
    print(f'  >>> VAL BPB: {bpb:.4f} | {p/1e6:.1f}M params | {train_time:.0f}s')
    return {'patch': patch_size, 'params': p, 'val_bpb': bpb, 'time': train_time,
            'seq_len': seq_len, 'n_patches': n_patches}

print('Harness ready')

In [ ]:
# ============================================================
# RUN ABLATIONS
# ============================================================

results = []
for ps in [2, 3, 4, 6, 8]:
    results.append(train_and_val(ps))

In [ ]:
# ============================================================
# RESULTS
# ============================================================

print()
print('='*70)
print('PATCH SIZE ABLATION RESULTS')
print('='*70)
print(f'{"Patch":>5} {"SeqLen":>6} {"Patches":>7} {"Params":>8} {"Val BPB":>9} {"Delta":>9} {"Time":>6}')
print('-'*70)

# Use patch=4 as reference (our current default)
ref_bpb = next(r['val_bpb'] for r in results if r['patch'] == 4)
best_bpb = min(r['val_bpb'] for r in results)

for r in results:
    d = r['val_bpb'] - ref_bpb
    tag = ''
    if r['patch'] == 4: tag = ' <-- current'
    elif r['val_bpb'] == best_bpb: tag = ' *** BEST'
    elif d < 0: tag = ' +'
    elif d > 0: tag = ' -'
    print(f'{r["patch"]:>5} {r["seq_len"]:>6} {r["n_patches"]:>7} {r["params"]/1e6:>7.1f}M '
          f'{r["val_bpb"]:>9.4f} {d:>+9.4f} {r["time"]:>5.0f}s{tag}')

print()
b = min(results, key=lambda r: r['val_bpb'])
print(f'Best: patch_size={b["patch"]} at {b["val_bpb"]:.4f} BPB')
print(f'vs current (patch=4): {ref_bpb - b["val_bpb"]:+.4f} BPB improvement')
print()
print('Note: smaller patch = more patches = more attention compute per step')
print('      larger patch = fewer patches = faster but coarser')
print('      patch_size affects param count via patch_proj and unpatch layers')